In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import classification_report

# Load data
X_train = pd.read_csv('../../dataset_processed/X_train_final.csv')
y_train = pd.read_csv('../../dataset_processed/y_train_final.csv').values.ravel()

In [2]:
# Define CV strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Initialize Baseline Model
model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

# Run cross-validation
scoring = ['f1_macro', 'recall_macro', 'accuracy']
cv_results = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

print(f"Mean CV Macro F1: {cv_results['test_f1_macro'].mean():.4f}")
print(f"Mean CV Macro Recall: {cv_results['test_recall_macro'].mean():.4f}")
print(f"Mean CV Accuracy: {cv_results['test_accuracy'].mean():.4f}")

# Fit on full data for classification report
model.fit(X_train, y_train)
y_pred = model.predict(X_train)
report = classification_report(y_train, y_pred)
print("\nClassification Report on Full Train Data:")
print(report)

import os
os.makedirs('results', exist_ok=True)
# Save metrics
with open('results/baseline_metrics.txt', 'w') as f:
    f.write(f"Mean CV Macro F1: {cv_results['test_f1_macro'].mean():.4f}\n")
    f.write(f"Mean CV Macro Recall: {cv_results['test_recall_macro'].mean():.4f}\n")
    f.write(f"Mean CV Accuracy: {cv_results['test_accuracy'].mean():.4f}\n\n")
    f.write("Classification Report on Full Train Data:\n")
    f.write(report)

Mean CV Macro F1: 0.4277
Mean CV Macro Recall: 0.4561
Mean CV Accuracy: 0.4873

Classification Report on Full Train Data:
              precision    recall  f1-score   support

           0       0.65      0.61      0.63     41193
           1       0.47      0.34      0.39     28211
           2       0.19      0.44      0.27      8950

    accuracy                           0.49     78354
   macro avg       0.44      0.46      0.43     78354
weighted avg       0.54      0.49      0.50     78354



## Phase 2: Hyperparameter Tuning with Optuna

In [3]:
import optuna
import warnings
from sklearn.exceptions import ConvergenceWarning

def objective(trial):
    # Hyperparameter search space
    # Solver selection: saga supports l1, l2, and elasticnet
    solver = trial.suggest_categorical('solver', ['saga', 'lbfgs'])
    
    if solver == 'saga':
        penalty = trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet'])
        if penalty == 'elasticnet':
            l1_ratio = trial.suggest_float('l1_ratio', 0.1, 0.9)
        else:
            l1_ratio = None
    else:  # lbfgs
        penalty = 'l2'
        l1_ratio = None
        
    C = trial.suggest_float('C', 1e-4, 50.0, log=True)
    tol = trial.suggest_float('tol', 1e-4, 1e-2, log=True)
    
    model = LogisticRegression(
        class_weight='balanced',
        max_iter=500,
        random_state=42,
        C=C,
        solver=solver,
        penalty=penalty,
        l1_ratio=l1_ratio,
        tol=tol
    )
    
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=ConvergenceWarning)
        warnings.filterwarnings('ignore', category=UserWarning)
        warnings.filterwarnings('ignore', category=FutureWarning)
        cv_results = cross_validate(model, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
        
    return cv_results['test_score'].mean()

# Disable optuna verbosity for clean logs
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
print("Starting Logistic Regression Deep Hyperparameter Tuning (30 trials)...")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"\nBest Params: {study.best_params}")
print(f"Best Macro F1: {study.best_value:.4f}")


[I 2026-10-11 02:37:26,087] A new study created in memory with name: no-name-4567d833-6dde-4930-b0bd-5e745f29e72e
[I 2026-10-11 02:37:32,124] Trial 0 finished with value: 0.4263353774767923 and parameters: {'C': 0.0074593432857265485, 'solver': 'lbfgs'}. Best is trial 0 with value: 0.4263353774767923.
[I 2026-10-11 02:37:37,120] Trial 1 finished with value: 0.4274484719523892 and parameters: {'C': 0.09846738873614563, 'solver': 'lbfgs'}. Best is trial 1 with value: 0.4274484719523892.
[I 2026-10-11 02:37:38,900] Trial 2 finished with value: 0.4095030010549305 and parameters: {'C': 0.00019517224641449495, 'solver': 'lbfgs'}. Best is trial 1 with value: 0.4274484719523892.
[W 2026-10-11 02:37:39,885] Trial 3 failed with parameters: {'C': 0.3470266988650412, 'solver': 'liblinear'} because of the following error: ValueError('\nAll the 5 fits failed.\nIt is very likely that your model is misconfigured.\nYou can try to debug the error by setting error_score=\'raise\'.\n\nBelow are more detai

ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "d:\EDUCATION\SLIIT\year 3\semester 1\ML\project\Healthcare_Readmission\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 860, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "d:\EDUCATION\SLIIT\year 3\semester 1\ML\project\Healthcare_Readmission\venv\Lib\site-packages\sklearn\base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\EDUCATION\SLIIT\year 3\semester 1\ML\project\Healthcare_Readmission\venv\Lib\site-packages\sklearn\linear_model\_logistic.py", line 1466, in fit
    raise ValueError(
ValueErr

## Phase 3: Tuned Model Evaluation

In [ ]:
# Initialize the tuned LogisticRegression using the best parameters
best_params = study.best_params.copy()
solver = best_params.get('solver', 'saga')
penalty = best_params.get('penalty', 'l2' if solver == 'lbfgs' else 'elasticnet')
l1_ratio = best_params.get('l1_ratio', None)
C = best_params['C']
tol = best_params.get('tol', 1e-4)

tuned_model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    solver=solver,
    penalty=penalty,
    l1_ratio=l1_ratio,
    C=C,
    tol=tol
)

# Run cross-validation
cv_results_tuned = cross_validate(tuned_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

mean_f1_tuned = cv_results_tuned['test_f1_macro'].mean()
print(f"Mean CV Macro F1 (Tuned): {mean_f1_tuned:.4f}")
print(f"Mean CV Macro Recall (Tuned): {cv_results_tuned['test_recall_macro'].mean():.4f}")
print(f"Mean CV Accuracy (Tuned): {cv_results_tuned['test_accuracy'].mean():.4f}")

# Fit on full training data
tuned_model.fit(X_train, y_train)
y_pred_tuned = tuned_model.predict(X_train)
report_tuned = classification_report(y_train, y_pred_tuned)
print("\nClassification Report on Full Train Data (Tuned):")
print(report_tuned)

# Save metrics
import os
os.makedirs('results', exist_ok=True)
with open('results/tuned_metrics.txt', 'w') as f:
    f.write(f"Mean CV Macro F1: {mean_f1_tuned:.4f}\n")
    f.write(f"Mean CV Macro Recall: {cv_results_tuned['test_recall_macro'].mean():.4f}\n")
    f.write(f"Mean CV Accuracy: {cv_results_tuned['test_accuracy'].mean():.4f}\n\n")
    f.write("Classification Report on Full Train Data (Tuned):\n")
    f.write(report_tuned)


## Phase 4: Model Comparison

In [ ]:
# Baseline vs Tuned Comparison
baseline_f1 = 0.4277
print("=== Logistic Regression Performance ===")
print(f"Baseline Macro F1: {baseline_f1:.4f}")
print(f"Tuned Macro F1:    {mean_f1_tuned:.4f}")
print(f"Improvement:       {mean_f1_tuned - baseline_f1:+.4f}")


## Phase 5: Save Final Model

In [ ]:
import joblib

# Persist the tuned model with best hyperparameters
joblib.dump(tuned_model, 'logistic_regression_best_model.pkl')
print("Model saved successfully to models/logistic_regression/logistic_regression_best_model.pkl")
